# ЛР 03.2 — GridSearchCV и финальный выбор модели (TODO)

## Цель
- взять feature set, выбранный в первом ноутбуке;
- честно подобрать гиперпараметры только на `train`;
- сравнить лучшие конфигурации на `validation`;
- один раз проверить `baseline_default` и `tuned_best` на `test`.

## Что важно
- `test` используется только в самом конце;
- preprocessing должен жить внутри `Pipeline`, а не обучаться заранее на всем наборе;
- итоговый выбор делается не по `train`, а по `validation`.


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd
from IPython.display import display
from sklearn.base import clone

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '03-overfitting-validation-and-hyperparameter-tuning',
    cwd.parent / '03-overfitting-validation-and-hyperparameter-tuning',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError(
        'Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 03 или из корня репозитория.'
    )

spec = importlib.util.spec_from_file_location('lab03_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

import json

from sklearn.model_selection import GridSearchCV, StratifiedKFold


## Шаг 1. Подготовка контекста для честного тюнинга

Для каждого датасета:
- повторяем тот же split `train/validation/test`;
- берем feature set, выбранный на основе `generalization_audit`;
- готовим контекст, который будет использоваться и в `GridSearchCV`, и в финальной проверке на `test`.


In [ ]:
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
generalization_audit = lab.load_generalization_audit()

split_context = {}
selection_rows = []

for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_valid, x_test, y_train, y_valid, y_test = lab.train_valid_test_split_stratified(x, y)

    feature_set_name = lab.choose_lab03_feature_set(generalization_audit, dataset_name)
    category_levels = lab.infer_category_levels(x_train)

    full_selector = lab.PreprocessedFeatureSelector(
        selected_features=None,
        category_levels=category_levels,
    ).fit(x_train, y_train)

    if feature_set_name == 'full':
        selected_features = None
        selected_feature_names = full_selector.get_feature_names_out().tolist()
    else:
        selected_features = feature_sets[dataset_name][feature_set_name]
        selected_feature_names = list(selected_features)

    split_context[dataset_name] = {
        'x_train': x_train,
        'x_valid': x_valid,
        'x_test': x_test,
        'y_train': y_train,
        'y_valid': y_valid,
        'y_test': y_test,
        'feature_set_name': feature_set_name,
        'selected_features': selected_features,
        'category_levels': category_levels,
    }

    selection_rows.append(
        {
            'dataset': dataset_name,
            'feature_set': feature_set_name,
            'n_train': len(x_train),
            'n_validation': len(x_valid),
            'n_test': len(x_test),
            'n_selected_features': len(selected_feature_names),
            'selected_preview': ', '.join(selected_feature_names[:5]),
        }
    )

chosen_feature_sets = pd.DataFrame(selection_rows)
chosen_feature_sets


## Шаг 2. GridSearchCV только на `train`

Здесь используется:
- `Pipeline`, чтобы preprocessing обучался внутри фолдов;
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`;
- multi-metric scoring по `f1`, `roc_auc`, `accuracy`;
- `refit='f1'` как главный критерий выбора.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
param_grids = lab.make_param_grids()

grid_frames = []
validation_rows = []
grid_cache = {}

for dataset_name, ctx in split_context.items():
    grid_cache[dataset_name] = {}

    for model_name, model in lab.make_tuning_models().items():
        pipeline = lab.build_model_pipeline(
            model=clone(model),
            selected_features=ctx['selected_features'],
            category_levels=ctx['category_levels'],
        )
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[model_name],
            scoring={
                'f1': 'f1',
                'roc_auc': 'roc_auc',
                'accuracy': 'accuracy',
            },
            refit='f1',
            cv=cv,
            n_jobs=1,
            return_train_score=False,
        )
        grid.fit(ctx['x_train'], ctx['y_train'])
        grid_cache[dataset_name][model_name] = grid

        cv_results = pd.DataFrame(grid.cv_results_)
        grid_frames.append(
            lab.top_gridsearch_rows(
                cv_results=cv_results,
                dataset_name=dataset_name,
                feature_set_name=ctx['feature_set_name'],
                model_name=model_name,
                top_n=5,
            )
        )

        validation_metrics = lab.evaluate_fitted_model(
            grid.best_estimator_,
            ctx['x_valid'],
            ctx['y_valid'],
        )
        validation_rows.append(
            {
                'dataset': dataset_name,
                'feature_set': ctx['feature_set_name'],
                'model': model_name,
                'validation_accuracy': validation_metrics['accuracy'],
                'validation_f1': validation_metrics['f1'],
                'validation_roc_auc': validation_metrics['roc_auc'],
                'best_params_json': json.dumps(
                    grid.best_params_,
                    ensure_ascii=False,
                    sort_keys=True,
                    default=str,
                ),
            }
        )

gridsearch_results_top = (
    pd.concat(grid_frames, ignore_index=True)
    .sort_values(['dataset', 'model', 'rank'])
    .reset_index(drop=True)
)
validation_summary = (
    pd.DataFrame(validation_rows)
    .sort_values(['dataset', 'validation_f1', 'validation_roc_auc'], ascending=[True, False, False])
    .reset_index(drop=True)
)

display(gridsearch_results_top.head(10))
validation_summary


## Шаг 3. Выбор финальной модели по `validation`

Правило выбора:
- максимум `validation f1`;
- затем максимум `validation roc_auc`;
- затем предпочесть `LogisticRegression`, если снова ничья.


In [ ]:
final_choice_rows = []

for dataset_name in sorted(split_context):
    winner = lab.choose_validation_winner(validation_summary, dataset_name)
    final_choice_rows.append(winner.to_dict())

final_choice_summary = (
    pd.DataFrame(final_choice_rows)
    .drop(columns=['model_priority'], errors='ignore')
    .reset_index(drop=True)
)
final_choice_summary


## Шаг 4. Один честный выход на `test`

После выбора конфигурации:
- обучаем `baseline_default` на `train + validation`;
- обучаем `tuned_best` на `train + validation`;
- сравниваем их только на `test`.

Это и есть тот момент, где можно обсуждать реальный выигрыш, а не промежуточную настройку.


In [ ]:
comparison_rows = []

for dataset_name, ctx in split_context.items():
    winner = final_choice_summary[final_choice_summary['dataset'] == dataset_name].iloc[0]
    model_name = winner['model']

    x_train_valid = pd.concat([ctx['x_train'], ctx['x_valid']], axis=0).reset_index(drop=True)
    y_train_valid = pd.concat([ctx['y_train'], ctx['y_valid']], axis=0).reset_index(drop=True)
    category_levels_train_valid = lab.infer_category_levels(x_train_valid)

    baseline_pipeline = lab.build_model_pipeline(
        model=clone(lab.make_default_models()[model_name]),
        selected_features=ctx['selected_features'],
        category_levels=category_levels_train_valid,
    )
    tuned_pipeline = lab.build_model_pipeline(
        model=clone(lab.make_tuning_models()[model_name]),
        selected_features=ctx['selected_features'],
        category_levels=category_levels_train_valid,
    )
    tuned_pipeline.set_params(**json.loads(winner['best_params_json']))

    _, baseline_metrics = lab.fit_and_evaluate_pipeline(
        estimator=baseline_pipeline,
        x_train=x_train_valid,
        y_train=y_train_valid,
        x_eval=ctx['x_test'],
        y_eval=ctx['y_test'],
    )
    _, tuned_metrics = lab.fit_and_evaluate_pipeline(
        estimator=tuned_pipeline,
        x_train=x_train_valid,
        y_train=y_train_valid,
        x_eval=ctx['x_test'],
        y_eval=ctx['y_test'],
    )

    for variant_name, metrics in [
        ('baseline_default', baseline_metrics),
        ('tuned_best', tuned_metrics),
    ]:
        comparison_rows.append(
            {
                'dataset': dataset_name,
                'feature_set': ctx['feature_set_name'],
                'model': model_name,
                'variant': variant_name,
                'accuracy': metrics['accuracy'],
                'f1': metrics['f1'],
                'roc_auc': metrics['roc_auc'],
                'fit_time_sec': metrics['fit_time_sec'],
            }
        )

baseline_vs_tuned_test_results = (
    pd.DataFrame(comparison_rows)
    .sort_values(['dataset', 'model', 'variant'])
    .reset_index(drop=True)
)
baseline_vs_tuned_test_results


## Самостоятельное изучение по ходу работы

Заполните своими словами:
- почему `GridSearchCV` нельзя настраивать по `test`;
- зачем preprocessing включается внутрь `Pipeline`;
- где tuned-модель действительно улучшилась, а где прибавка оказалась скромной.

При необходимости вынесите разбор в:
- `study-notes/gridsearchcv-practice.md`
- `study-notes/train-validation-test-split.md`


## Контрольные точки

Перед завершением ЛР проверьте:
1. `GridSearchCV` работал только на `train`.
2. `validation` использовался для выбора между лучшими конфигурациями.
3. `test` использован только один раз в финальном сравнении.
4. Сохранены таблицы `gridsearch_results_top` и `baseline_vs_tuned_test_results`.


In [ ]:
required_grid_columns = {
    'dataset',
    'feature_set',
    'model',
    'rank',
    'params_json',
    'mean_cv_f1',
    'std_cv_f1',
    'mean_cv_roc_auc',
    'mean_cv_accuracy',
    'mean_fit_time_sec',
}
required_test_columns = {
    'dataset',
    'feature_set',
    'model',
    'variant',
    'accuracy',
    'f1',
    'roc_auc',
    'fit_time_sec',
}

# TODO(обязательно):
# 1) Проверьте колонки в gridsearch_results_top и baseline_vs_tuned_test_results.
# 2) Сохраните оба DataFrame в CSV внутри outputs/.
# 3) Заполните narrative-блоки выше своими выводами.

raise NotImplementedError(
    'Самостоятельный блок не завершен: сохраните outputs/gridsearch_results_top.csv и outputs/baseline_vs_tuned_test_results.csv.'
)
